# MakharijPro AI — Track A / Phase 1+2: Environment Audit & Dataset Inventory

**Run this on Kaggle** (Settings → Accelerator: GPU T4x2 or P100, Internet: ON).

This notebook does NOT train anything. It only:
1. Records the exact environment (versions, GPU) so results are reproducible (SDD reproducibility requirement).
2. Inspects **QDAT** (small, fully downloadable — 1,505 clips) end-to-end.
3. Inspects **obadx/mualem-recitations-annotated** (~850 hours) in **streaming mode only** — we do NOT download
   850 hours in an audit pass. Kaggle's working directory has a 20GB persistent-output limit and this dataset
   alone would blow past it.

Output: `audit_report.json` — a machine-readable record of what was found, saved as a Kaggle notebook output
so you don't lose it when the session ends. Paste its contents (or just re-run and share errors) back to
continue to Phase 3 (validation) and Phase 4 (gap report).

## 0. Environment audit

In [ ]:
import sys, platform, subprocess, json, importlib

report = {"environment": {}}

report["environment"]["python_version"] = sys.version
report["environment"]["platform"] = platform.platform()

def pkg_version(name, import_name=None):
    try:
        mod = importlib.import_module(import_name or name)
        return getattr(mod, "__version__", "unknown")
    except ImportError:
        return "NOT INSTALLED"

for pkg, imp in [("tensorflow", "tensorflow"), ("librosa", "librosa"), ("numpy", "numpy"),
                 ("datasets", "datasets"), ("soundfile", "soundfile"), ("huggingface_hub", "huggingface_hub")]:
    report["environment"][pkg] = pkg_version(pkg, imp)

try:
    import tensorflow as tf
    gpus = tf.config.list_physical_devices('GPU')
    report["environment"]["gpu_count"] = len(gpus)
    report["environment"]["gpu_details"] = [str(g) for g in gpus]
except Exception as e:
    report["environment"]["gpu_check_error"] = str(e)

print(json.dumps(report["environment"], indent=2))

In [ ]:
# Install anything missing. On Kaggle these are usually preinstalled except `datasets`.
missing = [pkg for pkg, ver in report["environment"].items()
           if isinstance(ver, str) and ver == "NOT INSTALLED"]
if missing:
    print("Installing:", missing)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q"] + missing, check=True)
else:
    print("Nothing to install.")

## 1. QDAT — full inventory

QDAT is small (1,505 utterances) so we pull it completely and validate every file.
Labels cover 3 Tajweed rules: **Madd, Ghunnah, Ikhfaa** (correct/incorrect per rule).

In [ ]:
from datasets import load_dataset

qdat = load_dataset("obadx/qdat")
print(qdat)
print("\nFeatures:", qdat[list(qdat.keys())[0]].features)

In [ ]:
import numpy as np
import hashlib
from collections import Counter

def inventory_split(ds, split_name):
    rows = []
    durations = []
    sample_rates = Counter()
    channels = Counter()
    label_counts = Counter()
    corrupted = 0
    seen_hashes = {}
    duplicates = []

    for i, ex in enumerate(ds):
        try:
            audio = ex["audio"]
            arr = np.asarray(audio["array"], dtype=np.float32)
            sr = audio["sampling_rate"]
            dur = len(arr) / sr if sr else 0.0
            durations.append(dur)
            sample_rates[sr] += 1
            channels[arr.ndim] += 1

            h = hashlib.md5(arr.tobytes()).hexdigest()
            if h in seen_hashes:
                duplicates.append((seen_hashes[h], i))
            else:
                seen_hashes[h] = i

            # QDAT label field name is unknown until we see ex.keys() — capture whatever
            # non-audio fields exist so we can see the real schema.
            label_fields = {k: v for k, v in ex.items() if k != "audio"}
            for k, v in label_fields.items():
                label_counts[f"{k}={v}"] += 1

        except Exception as e:
            corrupted += 1
            print(f"[{split_name}] corrupted/unreadable example {i}: {e}")

    return {
        "split": split_name,
        "n_examples": len(ds),
        "n_corrupted": corrupted,
        "n_duplicates": len(duplicates),
        "duplicate_pairs_sample": duplicates[:10],
        "duration_seconds": {
            "min": float(np.min(durations)) if durations else None,
            "max": float(np.max(durations)) if durations else None,
            "mean": float(np.mean(durations)) if durations else None,
            "median": float(np.median(durations)) if durations else None,
        },
        "sample_rates": dict(sample_rates),
        "channel_dims": dict(channels),
        "label_field_distribution_top20": dict(Counter(label_counts).most_common(20)),
    }

qdat_report = {}
for split in qdat.keys():
    print(f"Inventorying QDAT split: {split} ({len(qdat[split])} examples)...")
    qdat_report[split] = inventory_split(qdat[split], split)

print(json.dumps(qdat_report, indent=2, default=str))

### 1b. What actually differs inside the 108 duplicate-audio pairs?

The first run found 108 exact-waveform duplicates. Two very different explanations are possible:
1. The same clip is genuinely repeated (a real data-quality defect) → dedupe before splitting, full stop.
2. The same underlying recording was annotated once per Tajweed rule, so "duplicates" are actually
   distinct labeled rows sharing one audio file → dropping them would silently delete real labels,
   and it also explains what `target` vs. `separate_tide` / `the_tight_noon` / `concealment` mean
   (whichever field differs between a pair is the row-specific rule/label; whichever stays constant
   is shared metadata like age/gender).

We don't guess — we print the actual rows.

In [ ]:
pairs_to_check = qdat_report["train"]["duplicate_pairs_sample"][:6]
duplicate_comparison = []
for i, j in pairs_to_check:
    ex_i = {k: v for k, v in qdat["train"][i].items() if k != "audio"}
    ex_j = {k: v for k, v in qdat["train"][j].items() if k != "audio"}
    diff_keys = [k for k in ex_i if ex_i.get(k) != ex_j.get(k)]
    duplicate_comparison.append({
        "pair": [i, j], "example_i": ex_i, "example_j": ex_j, "differing_fields": diff_keys,
    })
    print(f"Pair ({i}, {j}): differing fields = {diff_keys}")
    print(f"  [{i}] {ex_i}")
    print(f"  [{j}] {ex_j}\n")

## 2. obadx/mualem-recitations-annotated — streaming inspection only

This dataset is NOT a single `train` split — it's organised as one config per **moshaf** (a
complete recitation collection: one reciter reciting the whole Quran in one riwayah/style), named
`moshaf_0.0`, `moshaf_0.1`, `moshaf_1.0`, ... `moshaf_30.0`, plus two catalogue configs:
`moshaf_metadata` and `reciters_metadata`. Discovered this from the actual error Kaggle returned —
not assumed in advance.

Step 1: load the two metadata configs fully (they're small catalogue tables, not audio) so we can
see real column names instead of guessing. Step 2: pick ONE moshaf config to stream-sample based on
whatever the metadata actually shows — we still do not download all 850 hours.

In [ ]:
moshaf_metadata = load_dataset("obadx/mualem-recitations-annotated", "moshaf_metadata")
reciters_metadata = load_dataset("obadx/mualem-recitations-annotated", "reciters_metadata")

print("moshaf_metadata:", moshaf_metadata)
print("\nreciters_metadata:", reciters_metadata)

moshaf_split = list(moshaf_metadata.keys())[0]
moshaf_df = moshaf_metadata[moshaf_split].to_pandas()
reciters_split = list(reciters_metadata.keys())[0]
reciters_df = reciters_metadata[reciters_split].to_pandas()

print("\nmoshaf_metadata columns:", list(moshaf_df.columns))
print(moshaf_df.head(10).to_string())
print("\nreciters_metadata columns:", list(reciters_df.columns))
print(reciters_df.head(10).to_string())

In [ ]:
from itertools import islice

SAMPLE_SIZE = 200  # increase later once we know the schema is what we expect

# Pick the smallest moshaf by whatever size/duration/count column actually exists in the
# printed table above — do not hardcode a guessed column name. Fall back to moshaf_0.0 if
# no obviously-numeric "size" column is found.
size_col_candidates = [c for c in moshaf_df.columns
                        if any(k in c.lower() for k in ["duration", "hour", "size", "count", "num_", "n_"])]
chosen_config = "moshaf_0.0"
if size_col_candidates:
    col = size_col_candidates[0]
    try:
        smallest_row = moshaf_df.loc[moshaf_df[col].astype(float).idxmin()]
        # try to build the config name from an id-like column
        id_col_candidates = [c for c in moshaf_df.columns if "id" in c.lower() or "moshaf" in c.lower()]
        print(f"Auto-selection column used: {col!r}; smallest row:\n{smallest_row}")
        print(f"id-like columns available: {id_col_candidates}")
    except Exception as e:
        print("Auto-selection failed, falling back to moshaf_0.0:", e)
else:
    print("No obvious size/duration column found; defaulting to moshaf_0.0. "
          "Inspect moshaf_df above and set `chosen_config` manually if this is wrong.")

print(f"\nUsing config: {chosen_config}")

# streaming=True with no split= returns an IterableDatasetDict — this does NOT trigger a
# full download, unlike calling load_dataset(config) without streaming just to read .keys().
mualem_stream_dict = load_dataset("obadx/mualem-recitations-annotated", chosen_config, streaming=True)
mualem_split = list(mualem_stream_dict.keys())[0]
mualem_stream = mualem_stream_dict[mualem_split]
print(f"Available splits for {chosen_config}: {list(mualem_stream_dict.keys())} (using {mualem_split!r})")
print("Features:", mualem_stream.features)

sample = list(islice(mualem_stream, SAMPLE_SIZE))
print(f"\nPulled {len(sample)} streamed examples for inspection.")
if sample:
    print("\nExample keys:", list(sample[0].keys()))
    print("\nFirst example (non-audio fields):")
    print({k: v for k, v in sample[0].items() if k != "audio"})

In [ ]:
mualem_report = {"sample_size": len(sample)}

if sample:
    durations = []
    sample_rates = Counter()
    field_value_counts = Counter()

    for ex in sample:
        audio = ex.get("audio")
        if audio is not None:
            arr = np.asarray(audio["array"], dtype=np.float32)
            sr = audio["sampling_rate"]
            durations.append(len(arr) / sr if sr else 0.0)
            sample_rates[sr] += 1
        for k, v in ex.items():
            if k != "audio":
                field_value_counts[f"{k}={v}"] += 1

    mualem_report["duration_seconds_sample"] = {
        "min": float(np.min(durations)) if durations else None,
        "max": float(np.max(durations)) if durations else None,
        "mean": float(np.mean(durations)) if durations else None,
    }
    mualem_report["sample_rates_sample"] = dict(sample_rates)
    mualem_report["field_value_top20_sample"] = dict(Counter(field_value_counts).most_common(20))

print(json.dumps(mualem_report, indent=2, default=str))

### 2b. Full schema check — did the top-20 value summary hide any fields?

The aggregate summary above only keeps a "top 20 most common `field=value`" count, which silently
drops any field whose value is unique per row (e.g. a `sifat`/phonetic-transcript structure) — it
would never surface in a "most common value" count. Checking the actual key list and one full raw
example directly, so we don't conclude "no error annotations here" from an artifact of how we
summarized rather than from what's actually in the data.

In [ ]:
mualem_full_schema = {"keys": [], "first_example_non_audio": {}}
if sample:
    mualem_full_schema["keys"] = list(sample[0].keys())
    mualem_full_schema["first_example_non_audio"] = {
        k: (v if isinstance(v, (int, float, bool, type(None))) else str(v)[:500])
        for k, v in sample[0].items() if k != "audio"
    }
    print("Full key list:", mualem_full_schema["keys"])
    print("\nFull first example (non-audio, values truncated to 500 chars):")
    for k, v in mualem_full_schema["first_example_non_audio"].items():
        print(f"  {k}: {v}")
else:
    print("No sample pulled — nothing to inspect.")

### 2c. Persist the moshaf/reciters metadata tables into the report

These were only `print()`-ed earlier and would be lost once the Kaggle session ends. Saving the
real column names and a row preview so we can decide, with evidence, which moshaf configs (if any)
carry learner/error annotations vs. which are clean professional-Qari references.

In [ ]:
def df_preview(df, n=15):
    return {"columns": list(df.columns), "n_rows": len(df), "head": df.head(n).to_dict(orient="records")}

metadata_report = {
    "moshaf_metadata": df_preview(moshaf_df),
    "reciters_metadata": df_preview(reciters_df),
}
print(json.dumps(metadata_report, indent=2, default=str)[:3000], "...")

## 3. Save the audit report

This is the artifact we use to fill in the real CO-4 gap table and decide the Phase 5 manifest
strategy. Do not skip saving it — Kaggle sessions can be interrupted.

In [ ]:
report["qdat"] = qdat_report
report["qdat_duplicate_comparison"] = duplicate_comparison
report["mualem_recitations_annotated"] = mualem_report
report["mualem_full_schema_check"] = mualem_full_schema
report["mualem_metadata"] = metadata_report

with open("audit_report.json", "w") as f:
    json.dump(report, f, indent=2, default=str)

print("Saved audit_report.json")
print(json.dumps(report, indent=2, default=str)[:2000], "...")

## Next step (do not run yet)

Once this notebook has actually run on Kaggle and you've shared `audit_report.json` (or any errors)
back, Phase 3 (deeper validation) and Phase 4 (the real CO-4 gap table, filled with actual numbers
instead of placeholders) come next — building on whatever this audit actually finds, not on
assumptions about the datasets' structure.